# NYC Taxi Fare Prediction — Linear Regression

This notebook trains and evaluates the final Linear Regression model using the cleaned dataset produced by `01_exploration.ipynb`.

## 1. Imports and load processed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_clean = pd.read_parquet(
    "../data/processed/yellow_tripdata_2025-01_clean.parquet"
)

print("Shape:", df_clean.shape)


## 2. Define features and target

In [ ]:
features = [
    "trip_distance",
    "trip_duration",
    "avg_speed_mph",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "pickup_hour",
    "pickup_day",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend",
    "is_rush_hour"
]

X = df_clean[features]
y = df_clean["fare_amount"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())


## 3. Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)


## 4. Preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_features = [
    "trip_distance",
    "trip_duration",
    "avg_speed_mph",
    "passenger_count",
    "pickup_hour",
    "pickup_day",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend",
    "is_rush_hour"
]

categorical_features = [
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])


## 5. Linear Regression model

In [ ]:
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])


In [ ]:
model.fit(X_train, y_train)

print("Model trained!")


## 6. Predictions and evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


## 7. Prediction results

In [ ]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
})

results["residual"] = results["actual"] - results["predicted"]
results["abs_error"] = results["residual"].abs()

results.head()


## 8. Actual vs predicted

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    results["actual"],
    results["predicted"],
    alpha=0.2
)

plt.plot(
    [0, 300],
    [0, 300],
    linestyle="--"
)

plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Fare")

plt.tight_layout()
plt.show()


## 9. Residual analysis

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    results["actual"],
    results["residual"],
    alpha=0.2
)

plt.axhline(0, linestyle="--")

plt.xlabel("Actual Fare")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residuals vs Actual Fare")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    df_clean.loc[X_test.index, "trip_distance"],
    results["residual"],
    alpha=0.2
)

plt.axhline(0, linestyle="--")

plt.xlabel("Trip Distance")
plt.ylabel("Residual")
plt.title("Residuals vs Trip Distance")

plt.tight_layout()
plt.show()


## 10. Largest prediction errors

In [ ]:
results.nlargest(20, "abs_error")


In [ ]:
worst_indices = results.nlargest(20, "abs_error").index

df_clean.loc[
    worst_indices,
    [
        "fare_amount",
        "trip_distance",
        "trip_duration",
        "avg_speed_mph",
        "passenger_count",
        "RatecodeID",
        "PULocationID",
        "DOLocationID",
        "payment_type"
    ]
]


## 11. High-fare segment

In [ ]:
high_fares = results[results["actual"] > 200].copy()

print("Number of rides:", len(high_fares))
print("Percentage:", len(high_fares) / len(results) * 100)

high_fares.nlargest(20, "actual")


## 12. Final model summary

In [ ]:
print("Final Linear Regression Model")
print("-----------------------------")
print(f"Features: {len(features)}")
print(f"Training rows: {len(X_train):,}")
print(f"Testing rows: {len(X_test):,}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")
